# Experiment Orchestation

### Import Modules


In [ ]:
import requests
import json
from datetime import datetime, timedelta
import pytz

# API key placeholder (provide at runtime or via env var)
api_key = "Key <RIPE_ATLAS_API_KEY>"

### Mapping Anchor ID to Probe ID

In [ ]:
ALL_ANCHORS_DETAILS = "data/vantage_points/all_anchors_details.json"
ANCHOR_TO_PROBE_MAPPING = "data/anchor_probe_info/anchor_probe_info.json"


# Function to fetch probe information for a single anchor ID
def fetch_probe_info(anchor_id):
    url = f"https://atlas.ripe.net/api/v2/anchors/{anchor_id}/"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        probe_info = data.get("probe")
        return probe_info
    else:
        print(f"Failed to fetch probe info for Anchor ID {anchor_id}")
        return None


# Load the JSON file containing anchor IDs
with open(ALL_ANCHORS_DETAILS, "r") as file:
    anchor_ids = json.load(file)

# Dictionary to store anchor ID to probe info mapping
anchor_to_probe_info = {}

# Fetch probe info for each anchor ID
for anchor_id in anchor_ids.keys():
    probe_info = fetch_probe_info(int(anchor_id))
    if probe_info is not None:
        anchor_to_probe_info[anchor_id] = probe_info
import random


def divide_dict(dictionary):
    keys = list(dictionary.keys())
    random.shuffle(keys)

    # Calculate the size of each part
    part_size = 20

    # Divide the shuffled keys into four parts
    parts = [keys[i : i + part_size] for i in range(0, len(keys), part_size)]

    # Create four dictionaries using the shuffled keys
    divided_dicts = [{key: dictionary[key] for key in part} for part in parts]

    return divided_dicts


divided_dict = divide_dict(anchor_to_probe_info)
for i in range(4):
    # Write the result to a new JSON file
    with open(f"anchor_probe_info_{i}.json", "w") as output_file:
        json.dump(divided_dict[i], output_file, indent=4)

print(f"Probe info for anchor IDs fetched and saved to {ANCHOR_TO_PROBE_MAPPING}")

Probe info for anchor IDs fetched and saved to anchor_probe_info_i.json


### Defining Constants and Measurement Payload

In [ ]:
ITERATION = 3
INTERVAL = 7200  # Number of seconds in 4 hours
ANCHOR_TO_PROBE_MAPPING = f"anchor_probe_info_{ITERATION}.json"  # e.g., "anchor_probe_info_0.json"
PING_TARGETS_FILE = "data/targets/all_cdn_domains_new.json"  # e.g., "all_cdn_domains_new.json"


# Function to read JSON from a file
def read_json_file(file_path):
    with open(file_path, "r") as file:
        data = json.load(file)
    return data


anchor_to_probe_mapping = read_json_file(ANCHOR_TO_PROBE_MAPPING)
NUM_ANCHORS = len(anchor_to_probe_mapping)
# Define the request payload for RIPE Atlas probes
probes_request = {
    "type": "probes",
    "value": "",  # Placeholder for anchor IDs
    "requested": NUM_ANCHORS,
}

# Define the parameters for the ping measurement
ping_definition = {
    "target": "",  # Placeholder for the target domain
    "af": 4,
    "packets": 3,
    "size": 48,
    "description": "Test measurement 1",  # Change as required
    "interval": INTERVAL,
    "resolve_on_probe": True,
    "skip_dns_check": False,
    "include_probe_id": False,
    "is_public": False,
    "type": "ping",
}

# Create the overall measurement payload
measurement_payload = {
    "definitions": [],  # List to store measurement definitions
    "probes": [],  # List to store probe requests
    "is_oneoff": False,
    "bill_to": "<BILLING_EMAIL>",  # Placeholder email for RIPE Atlas billing
    "start_time": 0,  # Placeholder for start time (to be updated)
    "stop_time": 0,  # Placeholder for stop time (to be updated)
}

### Read Necessary JSON Files

In [68]:
# Read the anchor ID to probe mapping from a JSON file
anchor_probe_ids = list(anchor_to_probe_mapping.values())
comma_separated_anchors_string = ",".join(
    [str(x) for x in anchor_probe_ids]
)  # comma separated string of Probe IDs

# Read the JSON file containing ping targets (CDN domains)
ping_targets = read_json_file(PING_TARGETS_FILE)
# required_cdn_providers = ["Amazon Cloudfront", "Google", "Akamai", "Cloudflare", "Fastly"] # Hardcoded
ping_cdn_domains = []

### Make Payload

In [69]:
# Extract first 20 CDN domains from specified providers
for cdn_provider, cdn_domains in ping_targets.items():
    # if cdn_provider in required_cdn_providers:
    # for i in range(20):
    ping_cdn_domains.extend(cdn_domains)

# Configure the probes request with anchor IDs
probes_request["value"] = comma_separated_anchors_string
measurement_payload["probes"].append(
    probes_request.copy()
)  # COPY OBJECT WHEN APPENDING

# Iterate through CDN domains and configure measurement definitions
for target in ping_cdn_domains:
    ping_definition["target"] = target
    ping_definition["description"] = (
        "Measurement to " + target
    )  # Change description as required
    measurement_payload["definitions"].append(
        ping_definition.copy()
    )  # COPY OBJECT WHEN APPENDING

### Specify time to run


In [70]:
# Constants for the start date and time
YEAR = 2024  # Replace with the desired year
MONTH = 2  # Replace with the desired month
DAY = 27  # Replace with the desired day
HOUR = 0 if ITERATION < 2 else 1  # Replace with the desired hour
MINUTE = 0 if ITERATION % 2 == 0 else 30  # Replace with the desired minute
SECOND = 0  # Replace with the desired second

DAYS_TO_END = 10  # Replace
# BUFFER_MIN= 30*ITERATION    # 15 min buffer for final measurement

# Create a UTC datetime object with a specific timezone offset
UTC_TIMEZONE = pytz.UTC
start_datetime = datetime(YEAR, MONTH, DAY, HOUR, MINUTE, SECOND, tzinfo=UTC_TIMEZONE)
stop_datetime = start_datetime + timedelta(days=DAYS_TO_END)

# Update the start and stop times in the measurement payload
measurement_payload["start_time"] = start_datetime.timestamp()
measurement_payload["stop_time"] = stop_datetime.timestamp()

### Send Request 

In [ ]:
MEASUREMENT_IDS = f"measurement_ids_{ITERATION}.json"  # e.g., "measurement_ids_0.json"

# Create ping measurements for all the targets and send a POST request
url = "https://atlas.ripe.net/api/v2/measurements/"

# Send the POST request to create measurements
response = requests.post(
    url,
    json=measurement_payload,
    headers={"Content-Type": "application/json", "Authorization": api_key},
)

# Check the response status code and handle accordingly
if response.status_code == 201:
    measurement_id = response.json()["measurements"]
    print(measurement_id)  # Prints measurement IDs to console

    with open(MEASUREMENT_IDS, "w") as json_file:
        json.dump(response.json(), json_file, indent=4)
else:
    print("Error:", response.status_code, response.text)

[68054925, 68054926, 68054928, 68054929, 68054930, 68054931, 68054932, 68054933, 68054934, 68054935, 68054936, 68054937, 68054938, 68054939, 68054940, 68054941, 68054942, 68054943, 68054944, 68054945, 68054946, 68054947, 68054948, 68054949, 68054950, 68054951, 68054952, 68054953, 68054954, 68054955, 68054956, 68054957, 68054958, 68054959, 68054960, 68054961, 68054962, 68054963, 68054964, 68054965, 68054966, 68054967, 68054968, 68054969, 68054970, 68054971, 68054972, 68054973, 68054974, 68054975, 68054976, 68054977, 68054978, 68054979, 68054980, 68054981, 68054982, 68054983, 68054984, 68054985, 68054986, 68054987, 68054988, 68054989, 68054990, 68054991, 68054992, 68054993, 68054994, 68054995, 68054996, 68054997, 68054998, 68054999, 68055000, 68055001, 68055002, 68055003, 68055004, 68055005, 68055006, 68055007, 68055008, 68055009, 68055010, 68055011, 68055012, 68055013, 68055014, 68055015, 68055016, 68055017, 68055018, 68055019, 68055020, 68055021, 68055022, 68055023, 68055024, 68055025]

### Check Measurement Status

In [29]:
import requests
import json


# Function to check the status of specific measurements
def check_measurement_status(api_key, measurement_ids):
    statuses = {}

    for measurement_id in measurement_ids["measurements"]:
        url = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/"
        response = requests.get(
            url, headers={"Content-Type": "application/json", "Authorization": api_key}
        )

        if response.status_code == 200:
            measurement_info = response.json()
            status = measurement_info.get("status", "Unknown")
            statuses[measurement_id] = status
            print(f"Measurement {measurement_id} has status: {status}")
        else:
            print(f"Failed to retrieve status for measurement {measurement_id}.")

    return statuses


# Example usage:
# Replace 'your_api_key_here' with your actual API key
# api_key = 'your_api_key_here'

# Replace 'measurement_ids.json' with the actual JSON file containing your measurement IDs
with open(f"measurement_ids_{ITERATION}.json", "r") as json_file:
    measurement_ids = json.load(json_file)

measurement_statuses = check_measurement_status(api_key, measurement_ids)

# Save measurement statuses to a JSON file
with open(f"measurement_statuses_{ITERATION}.json", "w") as json_file:
    json.dump(measurement_statuses, json_file, indent=2)

print("Measurement statuses saved to 'measurement_statuses.json'.")

Measurement 68052968 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052969 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052970 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052971 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052972 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052973 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052974 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052975 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052976 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052977 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052978 has status: {'id': 0, 'name': 'Specified', 'when': 1708947011}
Measurement 68052979 has status: {'id': 0, 'name': 'Specified', 'when': 1708

### Stop All Measurement (run only when you mess up)

In [51]:
# import requests
# import json

# # Function to stop specific measurements
# def stop_measurements(api_key, measurement_ids):
#     for measurement_id in measurement_ids["measurements"]:
#         url = f'https://atlas.ripe.net/api/v2/measurements/{measurement_id}/'
#         response = requests.delete(url, headers={'Content-Type': 'application/json', 'Authorization': api_key})

#         if response.status_code == 204:
#             print(f"Measurement {measurement_id} has been stopped.")
#         else:
#             print(f"Failed to stop measurement {measurement_id}.")

# # Example usage:
# # Replace 'your_api_key_here' with your actual API key
# # api_key = 'your_api_key_here'

# # Replace 'measurements_to_stop.json' with the actual JSON file containing your measurement IDs to stop
# for i in range(4):
#     with open(f'measurement_ids_{i}.json', 'r') as json_file:
#         measurements_to_stop = json.load(json_file)
#         stop_measurements(api_key, measurements_to_stop)

Measurement 68053621 has been stopped.
Measurement 68053622 has been stopped.
Measurement 68053623 has been stopped.
Measurement 68053624 has been stopped.
Measurement 68053625 has been stopped.
Measurement 68053626 has been stopped.
Measurement 68053627 has been stopped.
Measurement 68053628 has been stopped.
Measurement 68053629 has been stopped.
Measurement 68053630 has been stopped.
Measurement 68053631 has been stopped.
Measurement 68053632 has been stopped.
Measurement 68053633 has been stopped.
Measurement 68053634 has been stopped.
Measurement 68053635 has been stopped.
Measurement 68053636 has been stopped.
Measurement 68053637 has been stopped.
Measurement 68053638 has been stopped.
Measurement 68053639 has been stopped.
Measurement 68053640 has been stopped.
Measurement 68053641 has been stopped.
Measurement 68053642 has been stopped.
Measurement 68053643 has been stopped.
Measurement 68053644 has been stopped.
Measurement 68053645 has been stopped.
Measurement 68053646 has 

### Download Results

In [ ]:
# List of measurement IDs you want to download

measurement_ids = read_json_file(MEASUREMENT_IDS)["measurements"]
# measurement_ids = [<MEASUREMENT_ID_1>, <MEASUREMENT_ID_2>]  # Example IDs

# Directory to save measurement results
output_directory = f"measurement_results_{ITERATION}/"  # e.g., "measurement_results_0/"

# Create the directory if it doesn't exist
import os

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# global download_count
download_count = 0


# Function to download measurement results by ID
def download_measurement_result(measurement_id):
    url = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/results/"
    headers = {"Accept": "application/json", "Authorization": api_key}

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        result = response.json()
        output_file = os.path.join(
            output_directory, f"measurement_{measurement_id}_results.json"
        )
        with open(output_file, "w") as file:
            json.dump(result, file, indent=4)

        global download_count
        download_count += 1
        print(
            f"{download_count} Downloaded results for Measurement ID {measurement_id} to {output_file}"
        )
    else:
        print(
            f"Failed to download results for Measurement ID {measurement_id}. Status code: {response.status_code}"
        )


# Download results for each measurement ID in the list
for measurement_id in measurement_ids:
    download_measurement_result(measurement_id)

1 Downloaded results for Measurement ID 68052538 to measurement_results_0/measurement_68052538_results.json
2 Downloaded results for Measurement ID 68052539 to measurement_results_0/measurement_68052539_results.json
3 Downloaded results for Measurement ID 68052540 to measurement_results_0/measurement_68052540_results.json
4 Downloaded results for Measurement ID 68052541 to measurement_results_0/measurement_68052541_results.json
5 Downloaded results for Measurement ID 68052542 to measurement_results_0/measurement_68052542_results.json
6 Downloaded results for Measurement ID 68052543 to measurement_results_0/measurement_68052543_results.json
7 Downloaded results for Measurement ID 68052544 to measurement_results_0/measurement_68052544_results.json
8 Downloaded results for Measurement ID 68052545 to measurement_results_0/measurement_68052545_results.json
9 Downloaded results for Measurement ID 68052546 to measurement_results_0/measurement_68052546_results.json
10 Downloaded results for Me